# Cross-Selling & Up-Selling via Embedding Similarity
## Office Products — Hybrid Embedding Demonstration
**Prerequisite:** Phase 1 & 2 of the recommender pipeline must have been run.  
**Inputs required:** `office_hybrid_item_embeddings.npy`, `office_full_item2idx.json`, `meta_Office_Products.jsonl.gz`, `Office_Products.jsonl.gz`

---
> This notebook demonstrates that the 128-dimensional hybrid embeddings (behavioural + semantic) support meaningful cross-sell and up-sell retrieval using cosine similarity alone — **no retraining needed**.

### Step 1: Imports

In [ ]:
import numpy as np
import json
import gzip
import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

print("Imports ready.")

### Step 2: Load Hybrid Embeddings & Item Mappings

In [ ]:
# ── Load 128-dim hybrid embeddings (output of Phase 2) ────────────────────
final_embs = np.load('office_hybrid_item_embeddings.npy')

# ── Load full vocabulary ───────────────────────────────────────────────────
with open('office_full_item2idx.json', 'r') as f:
    full_item2idx = json.load(f)

full_idx2item = {v: k for k, v in full_item2idx.items()}

print(f"Embeddings shape : {final_embs.shape}")
print(f"Vocabulary size  : {len(full_item2idx)} items (including <PAD> and <UNK>)")
print(f"Embedding dim    : {final_embs.shape[1]}")

### Step 3: Load Metadata — Titles, Brands, Categories

In [ ]:
# Stream metadata from gzipped JSONL — RAM-safe
CHUNK_SIZE    = 50_000
relevant_cols = ['asin', 'parent_asin', 'title', 'description', 'brand', 'categories']

chunks, chunk = [], []
with gzip.open('meta_Office_Products.jsonl.gz', 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        chunk.append({col: obj.get(col) for col in relevant_cols})
        if len(chunk) >= CHUNK_SIZE:
            chunks.append(pd.DataFrame(chunk))
            chunk = []
            print(f"  Loaded {i+1} rows...", end='\r')
    if chunk:
        chunks.append(pd.DataFrame(chunk))

meta_df = pd.concat(chunks, ignore_index=True)
del chunks, chunk

print(f"\nMetadata loaded: {len(meta_df)} rows")
print(f"Columns: {meta_df.columns.tolist()}")

### Step 4: Build Lookup Dictionaries — Titles & Categories

In [ ]:
item_titles     = {}   # asin -> display title (str)
item_categories = {}   # asin -> primary category (str)

for _, row in meta_df.iterrows():
    asin        = row.get('asin') or ''
    parent_asin = row.get('parent_asin') or asin
    title       = str(row.get('title',  '') or '').strip()
    brand       = str(row.get('brand',  '') or '').strip()
    cats        = row.get('categories') or []
    # First non-empty category string becomes the primary category
    primary_cat = next(
        (c.strip() for c in cats if isinstance(c, str) and c.strip()),
        'Office Products'
    )
    display = title if title else (brand if brand else asin)

    for key in {asin, parent_asin} - {''}:
        item_titles[key]     = display[:80]
        item_categories[key] = primary_cat

print(f"Titles available     : {len(item_titles)}")
print(f"Categories available : {len(item_categories)}")
print(f"Unique categories    : {len(set(item_categories.values()))}")
print()
for k in list(item_titles.keys())[:3]:
    print(f"  {k} | {item_titles[k][:50]} | {item_categories.get(k, '—')}")

### Step 5: Build Ratings Lookup from Review Data

In [ ]:
# Load reviews to compute per-item average ratings (rating metadata for up-sell)
MAX_REVIEW_ROWS = 100_000
review_chunks, chunk = [], []

with gzip.open('Office_Products.jsonl.gz', 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        pa  = obj.get('parent_asin')
        rt  = obj.get('rating')
        if pa and rt is not None:
            chunk.append({'parent_asin': pa, 'rating': float(rt)})
        if i >= MAX_REVIEW_ROWS:
            break
        if len(chunk) >= 50_000:
            review_chunks.append(pd.DataFrame(chunk))
            chunk = []
    if chunk:
        review_chunks.append(pd.DataFrame(chunk))

reviews_df   = pd.concat(review_chunks, ignore_index=True)
del review_chunks, chunk

avg_ratings = reviews_df.groupby('parent_asin')['rating'].mean()
item_ratings = avg_ratings.to_dict()

print(f"Ratings loaded for {len(item_ratings)} items")
vals = list(item_ratings.values())
if vals:
    print(f"  Min: {min(vals):.2f}  Max: {max(vals):.2f}  Avg: {sum(vals)/len(vals):.2f} ★")

### Step 6: Core Cosine Similarity Function
*(Efficient vectorised — no full N×N matrix)*

In [ ]:
def cosine_similarity_single(query_vec: np.ndarray,
                              emb_matrix: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between ONE query vector and ALL rows
    of emb_matrix using vectorised NumPy — O(N·D), not O(N²).

    Parameters
    ----------
    query_vec  : shape (D,)
    emb_matrix : shape (N, D)

    Returns
    -------
    similarities : shape (N,)  — one score per item
    """
    q_norm = query_vec / (np.linalg.norm(query_vec) + 1e-10)
    m_norm = emb_matrix / (np.linalg.norm(emb_matrix, axis=1, keepdims=True) + 1e-10)
    return m_norm @ q_norm    # dot product of L2-normalised vectors = cosine sim


def get_query_embedding(item_id: str) -> tuple:
    """Retrieve (idx, embedding_vector) for item_id; (None, None) if missing."""
    idx = full_item2idx.get(item_id)
    if idx is None:
        print(f"  [!] '{item_id}' not in vocabulary.")
        return None, None
    return idx, final_embs[idx]


print("Similarity utilities defined.")

### Step 7: Cross-Sell Retrieval Function

In [ ]:
def cross_sell(item_id: str,
               k: int = 10,
               min_sim: float = 0.30) -> list:
    """
    Cross-Selling: retrieve top-K similar items.

    Strategy
    --------
    - If category info is available, prefer items from a DIFFERENT category
      (complementary products). Falls back to same-category if not enough
      cross-category results exceed min_sim.
    - If no category info is available, return any top-K similar items.

    Parameters
    ----------
    item_id  : ASIN of the query item
    k        : number of recommendations to return
    min_sim  : minimum cosine similarity threshold

    Returns
    -------
    List of dicts: item_id, title, category, similarity
    """
    query_idx, query_vec = get_query_embedding(item_id)
    if query_vec is None:
        return []

    query_cat = item_categories.get(item_id, '')
    sims      = cosine_similarity_single(query_vec, final_embs)
    ranked    = np.argsort(sims)[::-1]     # descending

    results = []
    for idx in ranked:
        if int(idx) == query_idx:
            continue                        # skip self
        cand_id = full_idx2item.get(int(idx), '')
        if cand_id in ('<PAD>', '<UNK>', ''):
            continue
        sim = float(sims[idx])
        if sim < min_sim:
            break                           # sorted — no need to continue
        cat = item_categories.get(cand_id, '')
        results.append({
            'item_id'   : cand_id,
            'title'     : item_titles.get(cand_id, '(no title)')[:70],
            'category'  : cat or 'Office Products',
            'similarity': round(sim, 4),
            '_cross'    : bool(query_cat and cat and cat != query_cat),
        })
        if len(results) >= k * 4:           # gather extra pool to re-rank
            break

    # Prefer cross-category; fill with same-category if needed
    cross = [r for r in results if r['_cross']]
    same  = [r for r in results if not r['_cross']]
    final = (cross + same)[:k]
    for r in final:
        del r['_cross']
    return final


print("cross_sell() defined.")

### Step 8: Up-Sell Retrieval Function

In [ ]:
def up_sell(item_id: str,
            k: int = 10,
            min_sim: float = 0.30,
            min_rating_delta: float = 0.1) -> list:
    """
    Up-Selling: retrieve top-K similar items from the SAME category
    that have a higher average rating (premium alternatives).

    Strategy
    --------
    - Restrict candidates to the same category as the query item.
    - Prefer items with avg_rating > query_rating + min_rating_delta (upgrades).
    - If rating metadata is absent, fall back to pure similarity within same category.

    Parameters
    ----------
    item_id          : ASIN of the query item
    k                : number of recommendations to return
    min_sim          : minimum cosine similarity threshold
    min_rating_delta : minimum rating improvement to qualify as an upgrade

    Returns
    -------
    List of dicts: item_id, title, category, similarity, avg_rating, rating_gain
    """
    query_idx, query_vec = get_query_embedding(item_id)
    if query_vec is None:
        return []

    query_cat    = item_categories.get(item_id, '')
    query_rating = item_ratings.get(item_id, 0.0)
    sims         = cosine_similarity_single(query_vec, final_embs)
    ranked       = np.argsort(sims)[::-1]

    upgrades, same_cat = [], []

    for idx in ranked:
        if int(idx) == query_idx:
            continue
        cand_id = full_idx2item.get(int(idx), '')
        if cand_id in ('<PAD>', '<UNK>', ''):
            continue
        sim = float(sims[idx])
        if sim < min_sim:
            break

        cat    = item_categories.get(cand_id, '')
        rating = item_ratings.get(cand_id, 0.0)

        # Restrict to same category (or skip check when no category info exists)
        if query_cat and cat and cat != query_cat:
            continue

        entry = {
            'item_id'    : cand_id,
            'title'      : item_titles.get(cand_id, '(no title)')[:70],
            'category'   : cat or 'Office Products',
            'similarity' : round(sim, 4),
            'avg_rating' : round(rating, 2),
            'rating_gain': round(rating - query_rating, 2),
        }
        if rating >= query_rating + min_rating_delta:
            upgrades.append(entry)
        else:
            same_cat.append(entry)

        if len(upgrades) + len(same_cat) >= k * 4:
            break

    # Return genuine upgrades first, then same-category fill
    return (upgrades + same_cat)[:k]


print("up_sell() defined.")

### Step 9: Unified `get_recommendations()` API

In [ ]:
def get_recommendations(item_id: str,
                        mode: str = 'cross',
                        k: int = 10,
                        min_sim: float = 0.30,
                        min_rating_delta: float = 0.1) -> list:
    """
    Unified recommendation entry point.

    Parameters
    ----------
    item_id          : ASIN of the query item
    mode             : 'cross' → cross-sell (complementary, different category)
                       'up'    → up-sell    (premium, same category, higher rating)
    k                : number of results to return
    min_sim          : minimum cosine similarity (default 0.30)
    min_rating_delta : minimum rating gain for up-sell (default 0.1)

    Returns
    -------
    List of recommendation dicts
    """
    if mode == 'cross':
        return cross_sell(item_id, k=k, min_sim=min_sim)
    elif mode == 'up':
        return up_sell(item_id, k=k, min_sim=min_sim,
                       min_rating_delta=min_rating_delta)
    else:
        raise ValueError(f"mode must be 'cross' or 'up', got '{mode}'")


print("get_recommendations() API ready.")

### Step 10: Display Helper

In [ ]:
def display_recommendations(item_id: str, results: list, mode: str):
    """Pretty-print the recommendation list for a given item."""
    query_title  = item_titles.get(item_id, item_id)
    query_cat    = item_categories.get(item_id, 'Office Products')
    query_rating = item_ratings.get(item_id, None)
    label        = 'CROSS-SELL' if mode == 'cross' else 'UP-SELL'

    print("\n" + "=" * 72)
    print(f"  {label} RECOMMENDATIONS")
    print("=" * 72)
    print(f"  Input Item  : {query_title[:62]}")
    print(f"  Category    : {query_cat}")
    if query_rating:
        print(f"  Avg Rating  : {query_rating:.2f} \u2605")
    print(f"  ASIN        : {item_id}")
    print("-" * 72)

    if not results:
        print("  No recommendations found. Try lowering min_sim.")
    else:
        if mode == 'up':
            print(f"  {'#':<3} {'Title':<50} {'Sim':>6}  {'Rating':>7}  {'Gain':>6}")
            print("  " + "-" * 69)
            for i, r in enumerate(results, 1):
                gain_str   = f"+{r['rating_gain']:.2f}" if r['rating_gain'] > 0 else f"{r['rating_gain']:.2f}"
                rating_str = f"{r['avg_rating']:.2f} \u2605" if r['avg_rating'] > 0 else '   n/a'
                print(f"  {i:<3} {r['title']:<50} {r['similarity']:>6.4f}  {rating_str:>7}  {gain_str:>6}")
        else:
            print(f"  {'#':<3} {'Title':<50} {'Sim':>6}  {'Category'}")
            print("  " + "-" * 69)
            for i, r in enumerate(results, 1):
                print(f"  {i:<3} {r['title']:<50} {r['similarity']:>6.4f}  {r['category']}")
    print("=" * 72)


print("display_recommendations() ready.")

### Step 11: Select 5 Demo Items from the Vocabulary

In [ ]:
# Pick the 5 most-reviewed items that have titles in the vocabulary
freq = reviews_df['parent_asin'].value_counts()

demo_items = []
for asin, count in freq.items():
    if asin in full_item2idx and asin in item_titles:
        demo_items.append(asin)
    if len(demo_items) == 5:
        break

print("=" * 72)
print("  DEMO ITEMS SELECTED")
print("=" * 72)
for i, asin in enumerate(demo_items, 1):
    rating_str = f"{item_ratings.get(asin, 0):.2f} \u2605" if asin in item_ratings else 'n/a'
    print(f"  {i}. {item_titles.get(asin, asin)[:58]}")
    print(f"     ASIN: {asin} | Category: {item_categories.get(asin, '—')[:35]} | Rating: {rating_str}")
print("=" * 72)

### Step 12: Cross-Sell Demonstration — Top 10 for Each Demo Item

In [ ]:
print("\n" + "#" * 72)
print("  CROSS-SELL DEMONSTRATION")
print("  Strategy: similar items from complementary / different categories")
print("#" * 72)

for item_id in demo_items:
    results = get_recommendations(item_id, mode='cross', k=10)
    display_recommendations(item_id, results, mode='cross')

### Step 13: Up-Sell Demonstration — Top 10 for Each Demo Item

In [ ]:
print("\n" + "#" * 72)
print("  UP-SELL DEMONSTRATION")
print("  Strategy: same category, higher-rated alternatives")
print("#" * 72)

for item_id in demo_items:
    results = get_recommendations(item_id, mode='up', k=10)
    display_recommendations(item_id, results, mode='up')

### Step 14: Similarity Distribution Visualisation

In [ ]:
# Gather scores across all 5 demo items for both modes
cs_sims, us_sims, us_gains = [], [], []

for item_id in demo_items:
    for r in get_recommendations(item_id, mode='cross', k=10, min_sim=0.0):
        cs_sims.append(r['similarity'])
    for r in get_recommendations(item_id, mode='up', k=10, min_sim=0.0):
        us_sims.append(r['similarity'])
        if r['avg_rating'] > 0:
            us_gains.append(r['rating_gain'])

fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

def _hist(ax, data, color, title, xlabel, vlines=None):
    ax.hist(data, bins=20, color=color, alpha=0.85, edgecolor='white')
    if vlines:
        for x, lbl, c in vlines:
            ax.axvline(x, color=c, linestyle='--', linewidth=1.8, label=lbl)
        ax.legend(fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.grid(True, alpha=0.3)

if cs_sims:
    _hist(fig.add_subplot(gs[0]), cs_sims, 'steelblue',
          'Cross-Sell\nSimilarity Distribution', 'Cosine Similarity',
          vlines=[(np.mean(cs_sims), f'Mean={np.mean(cs_sims):.3f}', 'red')])

if us_sims:
    _hist(fig.add_subplot(gs[1]), us_sims, 'darkorange',
          'Up-Sell\nSimilarity Distribution', 'Cosine Similarity',
          vlines=[(np.mean(us_sims), f'Mean={np.mean(us_sims):.3f}', 'red')])

if us_gains:
    _hist(fig.add_subplot(gs[2]), us_gains, 'seagreen',
          'Up-Sell\nRating Gain Distribution', 'Rating Gain (stars)',
          vlines=[(0, 'No gain', 'gray')])

fig.suptitle(
    'Recommendation Similarity & Quality — Office Products (5 Demo Items)',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('office_crossupsell_distributions.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: office_crossupsell_distributions.png")

### Step 15: Coverage & Quality Summary

In [ ]:
print("\n" + "=" * 72)
print("  COVERAGE & QUALITY SUMMARY  (200-item random sample)")
print("=" * 72)

all_vocab = [k for k in full_item2idx if k not in ('<PAD>', '<UNK>')]
SAMPLE    = min(200, len(all_vocab))
random.seed(42)
sample_ids = random.sample(all_vocab, SAMPLE)

cs_covered, us_covered   = 0, 0
cs_sim_all, us_sim_all   = [], []

for item_id in sample_ids:
    cr = get_recommendations(item_id, mode='cross', k=5, min_sim=0.30)
    ur = get_recommendations(item_id, mode='up',    k=5, min_sim=0.30)
    if cr:
        cs_covered += 1
        cs_sim_all.extend(r['similarity'] for r in cr)
    if ur:
        us_covered += 1
        us_sim_all.extend(r['similarity'] for r in ur)

print(f"\n  Evaluated on {SAMPLE} randomly sampled vocabulary items")
print()
print("  Cross-Sell  (sim >= 0.30):")
print(f"    Items with >=1 recommendation : {cs_covered}/{SAMPLE} ({cs_covered/SAMPLE*100:.1f}%)")
if cs_sim_all:
    print(f"    Mean similarity              : {np.mean(cs_sim_all):.4f}")
    strong = sum(1 for s in cs_sim_all if s >= 0.70)
    print(f"    Strong hits (sim >= 0.70)    : {strong} results")
print()
print("  Up-Sell     (sim >= 0.30):")
print(f"    Items with >=1 recommendation : {us_covered}/{SAMPLE} ({us_covered/SAMPLE*100:.1f}%)")
if us_sim_all:
    print(f"    Mean similarity              : {np.mean(us_sim_all):.4f}")
    strong = sum(1 for s in us_sim_all if s >= 0.70)
    print(f"    Strong hits (sim >= 0.70)    : {strong} results")
print()
print("  Interpretation guide:")
print("    Coverage >= 70%   → retrieval is reliable across the catalogue")
print("    Mean sim  >= 0.50 → strong semantic alignment in embedding space")
print("    Strong    >= 50%  → most recommendations are high-confidence")
print("=" * 72)
print("  DEMONSTRATION COMPLETE")
print("=" * 72)